# NB11

Multi-tissue specificity across blood, spleen, liver, lung, bone marrow, and lymph node.

In [ ]:
# Multi-tissue specificity

import os, re, gc, glob, time, warnings, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.sparse as sp
from sklearn.decomposition import NMF

import matplotlib
matplotlib.rcParams.update({
    "font.family":"Arial","font.size":8,"axes.titlesize":9,"axes.labelsize":8,
    "xtick.labelsize":7,"ytick.labelsize":7,"legend.fontsize":7,"figure.dpi":150,
    "savefig.dpi":1200,"savefig.bbox":"tight","savefig.pad_inches":0.05,
    "axes.linewidth":0.8,"pdf.fonttype":42,"ps.fonttype":42,
})
import matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_style("ticks"); HAS_SNS=True
except ImportError:
    HAS_SNS=False
import scanpy as sc
warnings.filterwarnings("ignore"); np.random.seed(42); SEED=42

BASE_DIR = Path(os.environ.get("MES_BASE_DIR", "."))
RAW_DIR=BASE_DIR/"Raw Data"; PROC_DIR=BASE_DIR/"Process Data"
MANUSCRIPT_DIR=(BASE_DIR/"Manuscript data") if (BASE_DIR/"Manuscript data").exists() else (BASE_DIR/"Manuscript Data")
FIG_DIR=MANUSCRIPT_DIR/"Figures"/"Revision"; FIG_DIR.mkdir(parents=True,exist_ok=True)
TAB_DIR=MANUSCRIPT_DIR/"Tables"/"Revision"; TAB_DIR.mkdir(parents=True,exist_ok=True)
AIM2_DIR=PROC_DIR/"aim2_microglia"; NEG_DIR=PROC_DIR/"negctrl"
TS_FULL=RAW_DIR/"Thymus"/"GSE201333"/"GSM6058681_TabulaSapiens.h5ad"
BLOOD_QC=NEG_DIR/"TS_Blood_NegCtrl_Myeloid__qc.h5ad"

run_log=[]
def log(m):
    s=f"[{time.strftime('%H:%M:%S')}] {m}"; print(s,flush=True); run_log.append({"t":time.strftime('%H:%M:%S'),"m":m})
def save_fig(fig,name):
    out=FIG_DIR/f"Supp_Rev_{name}.png"; fig.savefig(out,dpi=1200,bbox_inches="tight"); plt.close(fig); log(f"FIG saved: {out.name}")
def save_xlsx(sheets,name):
    if isinstance(sheets,pd.DataFrame): sheets={"Sheet1":sheets}
    out=TAB_DIR/f"Supp_Rev_{name}.xlsx"
    with pd.ExcelWriter(out,engine="openpyxl") as w:
        for sn,df in sheets.items():
            (df if (df is not None and len(df)) else pd.DataFrame({"note":["no data"]})).to_excel(w,index=False,sheet_name=str(sn)[:31])
    log(f"TAB saved: {out.name}")

TOL_HOMEO=["P2RY12","CX3CR1","TMEM119","GPR34","SALL1","CSF1R","OLFML3"]
TOL_ACTIV=["APOE","SPP1","LPL","TREM2","CST7","CTSD","TYROBP","FCER1G","LGALS3","CD68"]

def looks_log1p(a,n=2000):
    X=a.X; v=(X.data[:min(X.data.size,n)] if X.data.size else np.array([0.])) if sp.issparse(X) else np.asarray(X).ravel()[:n]
    return (np.nanmax(v)<25) and (np.mean(np.abs(v-np.round(v))>1e-6)>0.2)
def umap(vn): return {str(v).upper():str(v) for v in vn}
def present(a,genes):
    m=umap(a.var_names); out=[]
    for g in genes:
        gg=str(g).upper()
        if gg in m and m[gg] not in out: out.append(m[gg])
    return out
def score(a,genes,name,min_g=3):
    pr=present(a,genes)
    if len(pr)<min_g: a.obs[name]=np.nan; return 0
    try: sc.tl.score_genes(a,pr,score_name=name,use_raw=False); return len(pr)
    except Exception: a.obs[name]=np.nan; return 0
def spearman_safe(x,y,minn=20):
    x=pd.to_numeric(pd.Series(x),errors="coerce").to_numpy(); y=pd.to_numeric(pd.Series(y),errors="coerce").to_numpy()
    m=np.isfinite(x)&np.isfinite(y)
    if m.sum()<minn: return np.nan,np.nan,int(m.sum())
    r,p=stats.spearmanr(x[m],y[m]); return float(r),float(p),int(m.sum())
def _X_dense(a,genes):
    X=a[:,genes].X
    return X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)

def to_symbols(a):
    sample=[str(v) for v in list(a.var_names[:50])]
    ens=sum(1 for v in sample if v.upper().startswith(("ENSG","ENSMUSG","ENS")))
    if ens < len(sample)*0.5: return "already_symbols"
    for col in ["feature_name","gene_symbol","gene_symbols","gene_name","symbol","Symbol","gene_short_name","hgnc_symbol","SYMBOL"]:
        if col in a.var.columns:
            syms=a.var[col].astype(str).values
            nonens=np.mean([not s.upper().startswith("ENS") and s.lower()!="nan" for s in syms[:200]])
            if nonens>0.5:
                a.var["_orig_id"]=a.var_names; a.var_names=pd.Index(syms); a.var_names_make_unique(); return col
    return None

# Substring tokens (safe: these strings do not lurk inside non-myeloid cell names)
MYELOID_TOKENS=["MACROPHAGE","MONOCYTE","DENDRITIC","MYELOID","MICROGLI","KUPFFER",
                "LANGERHANS","NEUTROPHIL","GRANULOCYTE","MAST CELL"]
# Standalone-word tokens (matched as whole words to avoid substring false positives, e.g. 'DC')
MYELOID_WORD_TOKENS={"DC","CDC","PDC","MO","MΦ"}
def is_myeloid_label(s):
    s=str(s).upper()
    if any(t in s for t in MYELOID_TOKENS): return True
    words=set(re.split(r"[^A-Z0-9Φ]+", s))
    return len(words & MYELOID_WORD_TOKENS)>0

# MES gene sets
log("="*72); log("LOAD: MES gene weights")
def _find_gw():
    for d in ["Manuscript data","Manuscript Data","Manuscript_Data","Manuscript","Process Data"]:
        for td in ["Tables","tables","Table",""]:
            for fn in ["Main_Table1.xlsx","Table1.xlsx"]:
                p=(BASE_DIR/d/td/fn) if td else (BASE_DIR/d/fn)
                if p.exists():
                    try:
                        df=pd.read_excel(p,sheet_name="GeneWeights")
                        if any(str(c).startswith("MES") for c in df.columns): return p,df
                    except Exception: pass
    for p in BASE_DIR.rglob("*Table1*.xlsx"):
        try:
            if "GeneWeights" in pd.ExcelFile(p).sheet_names:
                df=pd.read_excel(p,sheet_name="GeneWeights")
                if any(str(c).startswith("MES") for c in df.columns): return p,df
        except Exception: continue
    return None,None
MAIN_T1,df_weights=_find_gw()
if MAIN_T1 is None: raise FileNotFoundError("Main_Table1.xlsx not found")
mes_cols=[c for c in df_weights.columns if str(c).startswith("MES")]
mes_gene_sets={m: df_weights[["gene",m]].dropna().sort_values(m,ascending=False).head(50)["gene"].astype(str).str.upper().tolist() for m in mes_cols}
log(f"  {len(mes_cols)} thymus modules")

# Load microglia cohorts
log("="*72); log("LOAD: scored microglia cohorts")
cohorts={}
for f in sorted(glob.glob(str(AIM2_DIR/"*__microglia_scored.h5ad"))):
    ds=Path(f).name.replace("__microglia_scored.h5ad","")
    try:
        a=sc.read_h5ad(f); a.obs_names_make_unique()
        if "counts" not in a.layers: a.layers["counts"]=a.X.copy()
        if not looks_log1p(a):
            a.X=a.layers["counts"].copy(); sc.pp.normalize_total(a,target_sum=1e4); sc.pp.log1p(a)
        score(a,TOL_HOMEO,"_h"); score(a,TOL_ACTIV,"_a"); a.obs["tolerance_positioning"]=a.obs["_h"]-a.obs["_a"]
        for m in mes_cols:
            if f"{m}_score" not in a.obs.columns: score(a,mes_gene_sets[m],f"{m}_score")
        cohorts[ds]=a
        log(f"  {ds}: {a.n_obs:,} cells")
    except Exception as e:
        log(f"  {ds} failed: {e}")

# Module derivation helper (identical procedure for every tissue)
def derive_modules(adata_symbol, K=8, n_hvg=2500, prefix="MOD", cap=60000):
    """adata_symbol: already symbol-indexed, raw counts in .layers['counts'] or .X.
    Returns dict of prefixK -> top50 symbol genes. Identical pipeline for all tissues."""
    bt=adata_symbol.copy()
    if "counts" not in bt.layers: bt.layers["counts"]=bt.X.copy()
    bt.X=bt.layers["counts"].copy()
    sc.pp.normalize_total(bt,target_sum=1e4); sc.pp.log1p(bt)
    try:
        sc.pp.highly_variable_genes(bt,n_top_genes=n_hvg,flavor="seurat_v3")
    except Exception:
        sc.pp.highly_variable_genes(bt,n_top_genes=n_hvg)
    hvg=bt.var_names[bt.var["highly_variable"]].tolist()
    sym_frac=float(np.mean([not str(g).upper().startswith("ENS") for g in hvg])) if hvg else 0.0
    rng=np.random.RandomState(SEED)
    idx=rng.choice(bt.n_obs,size=min(bt.n_obs,cap),replace=False) if bt.n_obs>cap else np.arange(bt.n_obs)
    Xb=_X_dense(bt[idx,hvg], hvg)
    model=NMF(n_components=K, init="nndsvda", random_state=SEED, max_iter=800); model.fit(Xb)
    H=model.components_
    mods={}
    for k in range(K):
        order=np.argsort(H[k])[::-1][:50]
        mods[f"{prefix}{k+1:02d}"]=[str(hvg[i]).upper() for i in order]
    del bt; gc.collect()
    return mods, sym_frac

def coupling_for_modules(mods, label):
    rows=[]
    for ds,a in cohorts.items():
        for mk,genes in mods.items():
            score(a,genes,f"_{label}_{mk}")
            r,p,n=spearman_safe(a.obs[f"_{label}_{mk}"], a.obs["tolerance_positioning"])
            rows.append({"dataset":ds,"module_source":label,"module":mk,"r":r,"p":p,"n":n})
    return rows

def map_check(mods):
    test=next(iter(cohorts.values()))
    return float(np.mean([len(present(test,g)) for g in mods.values()]))

# ATTACK 1: clean blood re-derivation (symbol-only)
log("="*72); log("ATTACK 1: blood modules from symbol-converted genes")
all_rows=[]; tissue_meta=[]
try:
    blood=sc.read_h5ad(BLOOD_QC); blood.obs_names_make_unique()
    used=to_symbols(blood); blood.var_names_make_unique()
    bmods,bfrac=derive_modules(blood, K=8, n_hvg=2500, prefix="BLOOD")
    mc=map_check(bmods)
    log(f"  blood symbol conversion: {used}; HVG symbol fraction: {bfrac:.2f}; genes mapped: {mc:.0f}/50")
    if mc>=5:
        all_rows+=coupling_for_modules(bmods,"blood")
        tissue_meta.append({"tissue":"blood","symbol_fraction":bfrac,"genes_mapped":mc,"n_cells":int(blood.n_obs),"status":"ok"})
    else:
        tissue_meta.append({"tissue":"blood","symbol_fraction":bfrac,"genes_mapped":mc,"status":"untestable (mapping<5)"})
    # thymus reference (from MES gene sets directly)
    for ds,a in cohorts.items():
        for m in mes_cols:
            r,p,n=spearman_safe(a.obs[f"{m}_score"], a.obs["tolerance_positioning"])
            all_rows.append({"dataset":ds,"module_source":"thymus","module":m,"r":r,"p":p,"n":n})
    tissue_meta.append({"tissue":"thymus","symbol_fraction":1.0,"genes_mapped":50,"status":"reference (MES)"})
    del blood; gc.collect()
except Exception as e:
    log(f"  ATTACK 1 failed: {e}\n{traceback.format_exc()}")

# ATTACK 2: additional control tissues from Tabula Sapiens full
log("="*72); log("ATTACK 2: spleen/liver/lung myeloid controls from Tabula Sapiens")
WANT_TISSUES=["spleen","liver","lung","bone_marrow","lymph_node"]
try:
    if not TS_FULL.exists():
        log(f"  TabulaSapiens_full not found at {TS_FULL}; ATTACK 2 skipped")
    else:
        log(f"  loading {TS_FULL.name} (large; backed obs read)")
        TS=sc.read_h5ad(TS_FULL)
        TS.obs_names_make_unique()
        # detect tissue + cell_type columns
        tcol=None
        for c in ["organ_tissue","tissue","tissue_in_publication","organ","Tissue"]:
            if c in TS.obs.columns: tcol=c; break
        ctcol=None
        for c in ["cell_type","cell_ontology_class","free_annotation","cell_type_ontology_term_id","author_cell_type"]:
            if c in TS.obs.columns: ctcol=c; break
        log(f"  tissue col: {tcol}; cell_type col: {ctcol}")
        if tcol is None:
            log("  no tissue column; ATTACK 2 skipped")
        else:
            avail=pd.Series(TS.obs[tcol].astype(str).str.lower().unique())
            log(f"  available tissues (sample): {list(avail[:30])}")
            used_sym=to_symbols(TS); TS.var_names_make_unique()
            log(f"  TS symbol conversion: {used_sym}")
            for want in WANT_TISSUES:
                match=[t for t in TS.obs[tcol].astype(str).unique() if want in str(t).lower()]
                if not match:
                    tissue_meta.append({"tissue":want,"status":"not found in atlas"}); log(f"  {want}: not found"); continue
                sel=TS.obs[tcol].astype(str).isin(match)
                sub=TS[sel.values].copy()
                # cell filter: prefer compartment==immune (clean), then myeloid-label, then all.
                comp_col="compartment" if "compartment" in sub.obs.columns else None
                if comp_col is not None:
                    imm=sub.obs[comp_col].astype(str).str.lower().eq("immune")
                    if imm.sum()>=300:
                        sub=sub[imm.values].copy(); filt=f"immune compartment ({int(imm.sum())} cells)"
                    elif ctcol is not None and sub.obs[ctcol].astype(str).map(is_myeloid_label).sum()>=300:
                        mye=sub.obs[ctcol].astype(str).map(is_myeloid_label)
                        sub=sub[mye.values].copy(); filt=f"myeloid label ({int(mye.sum())} cells)"
                    else:
                        filt=f"all cells ({sub.n_obs}; <300 immune)"
                elif ctcol is not None:
                    mye=sub.obs[ctcol].astype(str).map(is_myeloid_label)
                    if mye.sum()>=300:
                        sub=sub[mye.values].copy(); filt=f"myeloid label ({int(mye.sum())} cells)"
                    else:
                        filt=f"all cells ({sub.n_obs}; <300 myeloid)"
                else:
                    filt=f"all cells ({sub.n_obs}; no celltype col)"
                if sub.n_obs<300:
                    tissue_meta.append({"tissue":want,"status":f"too few cells ({sub.n_obs})"}); log(f"  {want}: too few cells"); del sub; gc.collect(); continue
                mods,frac=derive_modules(sub, K=8, n_hvg=2500, prefix=want[:3].upper())
                mc=map_check(mods)
                log(f"  {want}: {filt}, symbol frac {frac:.2f}, genes mapped {mc:.0f}/50")
                if mc>=5:
                    all_rows+=coupling_for_modules(mods,want)
                    tissue_meta.append({"tissue":want,"symbol_fraction":frac,"genes_mapped":mc,"n_cells":int(sub.n_obs),"filter":filt,"status":"ok"})
                else:
                    tissue_meta.append({"tissue":want,"symbol_fraction":frac,"genes_mapped":mc,"filter":filt,"status":"untestable (mapping<5)"})
                del sub, mods; gc.collect()
        del TS; gc.collect()
except Exception as e:
    log(f"  ATTACK 2 failed: {e}\n{traceback.format_exc()}")

df_couple=pd.DataFrame(all_rows)
save_xlsx({"coupling_all_tissues":df_couple,"tissue_meta":pd.DataFrame(tissue_meta)}, "S1_MultiTissue_Coupling")

# Compare thymus vs each control + pooled
log("="*72); log("COMPARISON: thymus vs each control tissue")
cmp_rows=[]
try:
    thy=df_couple[df_couple["module_source"]=="thymus"]["r"].abs().dropna()
    for tis in [t for t in df_couple["module_source"].unique() if t!="thymus"]:
        ctl=df_couple[df_couple["module_source"]==tis]["r"].abs().dropna()
        if len(thy)>=5 and len(ctl)>=5:
            U,p=stats.mannwhitneyu(thy,ctl,alternative="greater")
            cmp_rows.append({"control":tis,"mean_abs_r_thymus":float(thy.mean()),"mean_abs_r_control":float(ctl.mean()),
                             "fold":float(thy.mean()/ctl.mean()) if ctl.mean()>0 else np.nan,
                             "MW_U":float(U),"MW_p":float(p),"n_control":len(ctl),
                             "thymus_stronger":bool(p<0.05 and thy.mean()>ctl.mean())})
    # pooled non-thymus
    pooled=df_couple[df_couple["module_source"]!="thymus"]["r"].abs().dropna()
    if len(thy)>=5 and len(pooled)>=5:
        U,p=stats.mannwhitneyu(thy,pooled,alternative="greater")
        cmp_rows.append({"control":"ALL_nonthymus_pooled","mean_abs_r_thymus":float(thy.mean()),
                         "mean_abs_r_control":float(pooled.mean()),
                         "fold":float(thy.mean()/pooled.mean()) if pooled.mean()>0 else np.nan,
                         "MW_U":float(U),"MW_p":float(p),"n_control":len(pooled),
                         "thymus_stronger":bool(p<0.05 and thy.mean()>pooled.mean())})
    df_cmp=pd.DataFrame(cmp_rows)
    save_xlsx({"thymus_vs_controls":df_cmp}, "S2_Thymus_vs_Controls")
    for _,r in df_cmp.iterrows():
        log(f"  thymus vs {r['control']}: {r['mean_abs_r_thymus']:.3f} vs {r['mean_abs_r_control']:.3f} "
            f"({r['fold']:.2f}x), MW p={r['MW_p']:.3g} -> {'thymus stronger' if r['thymus_stronger'] else 'NOT distinguishable'}")
except Exception as e:
    log(f"  comparison failed: {e}\n{traceback.format_exc()}")
    df_cmp=pd.DataFrame()

# ATTACK 3: parameter robustness (K and HVG sweep, thymus vs blood)
log("="*72); log("ATTACK 3: parameter robustness sweep (K, HVG) thymus vs blood")
sweep_rows=[]
try:
    blood=sc.read_h5ad(BLOOD_QC); blood.obs_names_make_unique(); to_symbols(blood); blood.var_names_make_unique()
    thy_absr_ref=df_couple[df_couple["module_source"]=="thymus"]["r"].abs().dropna()
    for K in [6,8,10]:
        for nh in [1500,2500,3500]:
            try:
                bmods,bfrac=derive_modules(blood,K=K,n_hvg=nh,prefix=f"B{K}_{nh}_")
                mc=map_check(bmods)
                if mc<5:
                    sweep_rows.append({"K":K,"HVG":nh,"status":f"blood mapping<5 ({mc:.0f})"}); continue
                brows=coupling_for_modules(bmods,f"blood_K{K}_H{nh}")
                bdf=pd.DataFrame(brows); babs=bdf["r"].abs().dropna()
                if len(babs)>=5 and len(thy_absr_ref)>=5:
                    U,p=stats.mannwhitneyu(thy_absr_ref,babs,alternative="greater")
                    sweep_rows.append({"K":K,"HVG":nh,"mean_abs_r_blood":float(babs.mean()),
                                       "mean_abs_r_thymus_ref":float(thy_absr_ref.mean()),
                                       "fold":float(thy_absr_ref.mean()/babs.mean()) if babs.mean()>0 else np.nan,
                                       "MW_p":float(p),"thymus_stronger":bool(p<0.05),"blood_symbol_frac":bfrac,"status":"ok"})
                    log(f"  K={K}, HVG={nh}: fold={thy_absr_ref.mean()/babs.mean():.2f}x, p={p:.3g}")
                del bmods,brows,bdf; gc.collect()
            except Exception as ee:
                sweep_rows.append({"K":K,"HVG":nh,"status":f"failed: {ee}"})
    del blood; gc.collect()
    df_sweep=pd.DataFrame(sweep_rows)
    save_xlsx({"param_robustness":df_sweep}, "S3_Param_Robustness")
    ok=df_sweep[df_sweep.get("status","")=="ok"] if len(df_sweep) else pd.DataFrame()
    if len(ok):
        n_strong=int(ok["thymus_stronger"].sum())
        log(f"  thymus stronger in {n_strong}/{len(ok)} parameter settings; fold range "
            f"{ok['fold'].min():.2f}-{ok['fold'].max():.2f}")
except Exception as e:
    log(f"  ATTACK 3 failed: {e}\n{traceback.format_exc()}")
    df_sweep=pd.DataFrame()

# VERDICT
log("="*72); log("OVERALL VERDICT")
try:
    verdict={}
    if len(df_cmp):
        controls_tested=df_cmp[df_cmp["control"]!="ALL_nonthymus_pooled"]
        n_ctl=len(controls_tested); n_beat=int(controls_tested["thymus_stronger"].sum())
        pooled_row=df_cmp[df_cmp["control"]=="ALL_nonthymus_pooled"]
        verdict["n_controls_tested"]=n_ctl
        verdict["n_controls_thymus_beat"]=n_beat
        verdict["controls"]="; ".join(f"{r['control']}({r['fold']:.1f}x,p={r['MW_p']:.1e})" for _,r in controls_tested.iterrows())
        if len(pooled_row):
            verdict["pooled_fold"]=float(pooled_row["fold"].iloc[0])
            verdict["pooled_p"]=float(pooled_row["MW_p"].iloc[0])
        # robustness
        if len(df_sweep):
            ok=df_sweep[df_sweep.get("status","")=="ok"]
            if len(ok):
                verdict["param_settings_thymus_stronger"]=f"{int(ok['thymus_stronger'].sum())}/{len(ok)}"
                verdict["fold_range_across_params"]=f"{ok['fold'].min():.2f}-{ok['fold'].max():.2f}"
        # call
        if n_beat==n_ctl and n_ctl>=2:
            verdict["CALL"]=("PREFERENTIAL coupling robust: thymus modules couple more strongly than "
                             "ALL control tissues tested, across parameter settings. Strong, defensible, "
                             "reviewer-resistant. Frame as 'preferential' (fold-enrichment), not 'exclusive'.")
        elif n_beat>=1:
            # check lymphoid tissues specifically (spleen, lymph_node) for the specificity question
            lymphoid=controls_tested[controls_tested["control"].str.lower().str.contains("spleen|lymph")]
            lymph_beaten=int(lymphoid["thymus_stronger"].sum()) if len(lymphoid) else 0
            lymph_total=len(lymphoid)
            if lymph_total>0 and lymph_beaten<lymph_total:
                verdict["CALL"]=(f"PARTIAL/LYMPHOID: thymus beats {n_beat}/{n_ctl} controls but does NOT exceed "
                                 f"all lymphoid tissues ({lymph_beaten}/{lymph_total} lymphoid beaten). The effect "
                                 f"is LYMPHOID-associated, not thymus-exclusive. Narrow claim to 'lymphoid/thymic "
                                 f"myeloid programs'. This is honest and still novel.")
            else:
                verdict["CALL"]=(f"PARTIAL: thymus beats {n_beat}/{n_ctl} controls (incl lymphoid). Report the full "
                                 f"ranking; preferential coupling holds for tested tissues.")
        else:
            verdict["CALL"]=("NOT preferential: thymus does not significantly exceed controls. The coupling is "
                             "a generic myeloid-module property. Disclose; drop the specificity claim.")
    else:
        verdict["CALL"]="No comparison computed (insufficient tissues)."
    save_xlsx({"verdict":pd.DataFrame([verdict])}, "S_VERDICT")
    for k,v in verdict.items(): log(f"  {k}: {v}")
    # figure: mean |r| by tissue
    if len(df_couple):
        order=df_couple.groupby("module_source")["r"].apply(lambda s:s.abs().mean()).sort_values(ascending=False)
        fig,ax=plt.subplots(figsize=(5.0,3.4))
        colors=["#D6604D" if t=="thymus" else "#4393C3" for t in order.index]
        ax.bar(range(len(order)), order.values, color=colors, edgecolor="black", linewidth=0.4)
        ax.set_xticks(range(len(order))); ax.set_xticklabels(order.index, rotation=30, ha="right")
        ax.set_ylabel("mean |MES-tolerance coupling|"); ax.set_title("Tissue specificity: coupling by source tissue")
        for s in ["top","right"]: ax.spines[s].set_visible(False)
        save_fig(fig,"S2_Coupling_By_Tissue")
except Exception as e:
    log(f"  verdict failed: {e}\n{traceback.format_exc()}")

save_xlsx({"run_log":pd.DataFrame(run_log)}, "MASTER_NB15_RunLog")
log("="*72); log("NB15 COMPLETE"); log(f"  Tables: {TAB_DIR}")
